In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_core.messages import HumanMessage
import secret_key

## Load the model and secret key

In [9]:
api_key = secret_key.GOOGLE_API_KEY
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",api_key=api_key)

## Play with the model 

In [10]:
# Simple text invocation
result = llm.invoke("Give me a joke.")
print(result.content)

Why don't scientists trust atoms?

Because they make up everything!


## Let's load the csv file

In [ ]:
# Fix for UnicodeDecodeError: Specify the correct encoding
from langchain_community.document_loaders.csv_loader import CSVLoader

# Load CSV with Windows-1252 encoding (detected automatically)
loader = CSVLoader(
    file_path='codebasics_faqs.csv', 
    source_column='prompt',
    encoding='windows-1252'  # Specify the correct encoding
)

# Load the data
data = loader.load()
print(f"Successfully loaded {len(data)} documents from CSV!")

Successfully loaded 75 documents from CSV!


## Let's create Embedding 

In [36]:
'''
There are many embedding tools but some of them are not free
In this project we use HuggingFaceIntructEmbedding
Because it is free but good performing!
''' 

from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import FAISS
instructor_embeddings = HuggingFaceInstructEmbeddings()
vector_db = FAISS.from_documents(documents=data, embedding = instructor_embeddings)




/var/folders/f8/6cl1l5650fdg413kw5mh34jr0000gn/T/ipykernel_17028/1983548092.py:9: LangChainDeprecationWarning: Default values for HuggingFaceInstructEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceInstructEmbeddings constructor instead.
  instructor_embeddings = HuggingFaceInstructEmbeddings()
No sentence-transformers model found with name hkunlp/instructor-large. Creating a new one with mean pooling.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


In [ ]:
retriever = vector_db.as_retriever()
rdocs = retriever.invoke("For how long this course is vaid?")

`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


[Document(id='74e5054a-1ee2-4dca-9fdb-c3cac287701a', metadata={'source': 'Once purchased, is this course available for lifetime access?', 'row': 22}, page_content='prompt: Once purchased, is this course available for lifetime access?\nresponse: Yes'),
 Document(id='e484918a-67dd-4657-bdb3-c04a79452448', metadata={'source': 'What is the duration of this bootcamp? How long will it last?', 'row': 8}, page_content='prompt: What is the duration of this bootcamp? How long will it last?\nresponse: You can complete all courses in 3 months if you dedicate 2-3 hours per day.'),
 Document(id='1920f205-3e2a-4803-ab99-97c7c5bf5522', metadata={'source': 'How can I get help if I have a doubt and need support?', 'row': 23}, page_content='prompt: How can I get help if I have a doubt and need support?\nresponse: We have an active discord server where you can post your question and most of the time you will get an answer in a reasonable time frame.'),
 Document(id='ef385746-8b1d-411e-856f-5c18f699d3a2', 

In [60]:
chain('Do you guys provide internship and also do you offer EMI payments?')

/var/folders/f8/6cl1l5650fdg413kw5mh34jr0000gn/T/ipykernel_17028/1022339835.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain('Do you guys provide internship and also do you offer EMI payments?')
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


{'query': 'Do you guys provide internship and also do you offer EMI payments?',
 'result': 'Yes, we provide virtual internships. No, we do not offer EMI payments.',
 'source_documents': [Document(id='08520dd0-0248-4af6-8369-afee559501a6', metadata={'source': 'Do you provide any virtual internship?', 'row': 14}, page_content='prompt: Do you provide any virtual internship?\nresponse: Yes'),
  Document(id='68c6c75e-45b6-4ec4-bafc-f5e5042d2896', metadata={'source': 'Do you provide any job assistance?', 'row': 11}, page_content='prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'),
  Document(id='87116c9f-a7ed-4643-a566-4feaff429928', metadata={'source': 'Do we have an EMI option?', 'row': 13}, page_content='prompt: Do we have an EMI option?\nresponse: No'),
  Document(id='1920f205-3e2a-4803-ab99-97c7c5bf

In [63]:
from langchain_classic.prompts import PromptTemplate
prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""


PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}

In [64]:
# NEW (correct import for LangChain 1.0+):
# Try this import path:
from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm,
chain_type = 'stuff',
retriever=retriever,
input_key='query',
return_source_documents=True,
chain_type_kwargs=chain_type_kwargs)


In [66]:
chain('Do we learn Javascripts as well?')

`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


{'query': 'Do we learn Javascripts as well?',
 'result': "I don't know.",
 'source_documents': [Document(id='ef385746-8b1d-411e-856f-5c18f699d3a2', metadata={'source': 'I use tableau, can I take this course?', 'row': 28}, page_content='prompt: I use tableau, can I take this course?\nresponse: Yes, you will still benefit from the concepts outside the tools that are discussed in this course such as business context, problem solving, project management tools, etc.'),
  Document(id='1920f205-3e2a-4803-ab99-97c7c5bf5522', metadata={'source': 'How can I get help if I have a doubt and need support?', 'row': 23}, page_content='prompt: How can I get help if I have a doubt and need support?\nresponse: We have an active discord server where you can post your question and most of the time you will get an answer in a reasonable time frame.'),
  Document(id='68c6c75e-45b6-4ec4-bafc-f5e5042d2896', metadata={'source': 'Do you provide any job assistance?', 'row': 11}, page_content='prompt: Do you provi